# Traceprop-LLM — LDS quality on frozen GPT-2 features (workstream C, tighten)

**Claim:** last-layer attribution recovers strong, correct influence on a real pretrained LLM. Frozen GPT-2 backbone + trained linear head on SST-2; Linear Datamodeling Score over random 50% data subsets. Traceprop-LL (exact last-layer grads) vs random.

GPU runtime. ~15-20 min for 500 subsets. Prints the LDS table at the end.

In [ ]:
!pip -q install transformers datasets scikit-learn
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .

## Run: frozen GPT-2 + linear head LDS (500 subsets)

In [ ]:
%cd /content/Traceprop/experiments
!python exp28_frozen_lds.py --backend hf --model gpt2 --device cuda \
    --n_train 3000 --n_test 500 --n_subsets 500 --seq 64 --proj_dim 512 --C 0.5

## Result

`traceprop_ll_trak` is the headline (currently 0.45 ± 0.06 at 256 subsets). 500 subsets should tighten the ±. `random` should sit at ~0.00. Paste the table back.

In [ ]:
import json, glob
for p in glob.glob('/content/Traceprop/experiments/results/exp28_hf_*.json'):
    d = json.load(open(p))
    print(d['model'], 'acc', d['target_test_acc'], 'subsets', d['n_subsets'], 'n_train', d['n_train'])
    for k, v in d['lds'].items():
        print(f"  {k:<20} {v['mean']:+.4f} ± {v['std']:.4f}")